In [20]:
# pip install Wikipedia-API
# !pip install haystack-ai
# !pip install transformers[torch,sentencepiece]

# get the snippes and data

In [21]:
import pandas as pd
import ast  # For safely evaluating strings containing Python expressions

def parse_list_string(s):
    try:
        # Safely evaluate the string representation of the list
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return []

# # Read the CSV file
df = pd.read_csv("combined_output.csv")

# Parse the string representation of lists in the scraping_and_procesor column
df["snippes_google_v2"] = df["snippes_google_v2"].apply(parse_list_string)


def fix_dict_to_list(list_of_dicts):
    lista=[]
    for dicts in list_of_dicts:
        if dicts["snippet"]:
            lista.append(dicts["snippet"])
    return lista


df["scraping_and_procesor"] = df["snippes_google_v2"].apply(fix_dict_to_list)

In [22]:
from haystack import Document
from haystack.components.readers import ExtractiveReader
import pandas as pd

def haystack_ranking(row):
    """
    Rank documents using Haystack's ExtractiveReader based on query relevance
    
    Args:
        row: DataFrame row containing 'query' and 'scraping_and_procesor' columns
        
    Returns:
        list: Sorted list of dictionaries containing chunks and their scores
    """
    # Initialize ExtractiveReader
    reader = ExtractiveReader(model="deepset/roberta-base-squad2")
    reader.warm_up()
    
    query = row['query']
    result = []
    
    # Process each chunk
    for chunk in row['scraping_and_procesor']:
        try:
            # Create Document object
            doc = Document(content=chunk)
            
            # Get prediction from reader
            prediction = reader.run(query=query, documents=[doc])
            
            # Extract score from prediction
            # If no answer is found, use 0 as score
            score = prediction['answers'][0].score if prediction.get('answers') else 0
            
            result.append({
                "chuck": chunk,
                "score": score
            })
            
        except Exception as e:
            print(f"Error processing chunk: {e}")
            continue
    
    # Sort results by score in descending order
    sorted_data = sorted(result, key=lambda x: x['score'], reverse=True)
    return sorted_data

# Apply ranking to DataFrame
df["haystack_ranking"] = df.apply(haystack_ranking, axis=1)

In [25]:
df["haystack_ranking"][0]

[{'chuck': 'Stats of Alberto Fouillioux ; World Cup qualification South America, World Cup qualification · 6 · 2, - ; World Cup 1966, World Cup · 4 · -, -\xa0...',
  'score': 0.6882550716400146},
 {'chuck': 'Nov 14, 2024 — He was part of two championship winning sides in 1961 and 1966. He played for Chile in two World Cups; the 1962 and 1966. In 1969, he joined\xa0...',
  'score': 0.648784339427948},
 {'chuck': 'He was part of two championship winning sides in 1961 and 1966. He played for Chile in two World Cups; the 1962 and 1966.',
  'score': 0.6088418960571289},
 {'chuck': 'Alberto Jorge Fouilloux Ahumada was a Football player. Born in Santiago ... - · -. EDITIONS. COMPETITION, G, Goals, AST. World Cup 1966. 2, 0 · 0.',
  'score': 0.6082761287689209},
 {'chuck': 'Titles and season. 2x World Cup participant. 1966, Chile. 1962, Chile. 1x Footballer of the Year. 1964, Chile. 1x World Cup third place. 1962, Chile.',
  'score': 0.5630112886428833},
 {'chuck': 'Alberto Fouillioux - Intern

In [29]:
def eliminate_not_significated_chunks(lista):
    """Eliminar los chunks que no tienen relevancia más de 60%, si no return none"""
    result = []
    for dictionary in lista: 
        chuck = dictionary['chuck'] 
        score = dictionary['score'] 
        if score > 0.45:
            result.append(chuck)
    result=result[:3]
    return result

# df["chunks_filter"] = df["chunk_ranking"].apply(eliminate_not_significated_chunks)
df["chunks_filter_hay"] = df["haystack_ranking"].apply(eliminate_not_significated_chunks)

In [30]:
df["chunks_filter_hay"][0]

['Stats of Alberto Fouillioux ; World Cup qualification South America, World Cup qualification · 6 · 2, - ; World Cup 1966, World Cup · 4 · -, -\xa0...',
 'Nov 14, 2024 — He was part of two championship winning sides in 1961 and 1966. He played for Chile in two World Cups; the 1962 and 1966. In 1969, he joined\xa0...',
 'He was part of two championship winning sides in 1961 and 1966. He played for Chile in two World Cups; the 1962 and 1966.']

# build the prompts

In [31]:
def add_token_positions(text):
    """
    Takes a string and returns it with each word wrapped in tags showing their start and end positions.
    """
    result = []
    in_word = False
    word_start = 0
    
    for i, char in enumerate(text):
        if not char.isspace() and not in_word:
            in_word = True
            word_start = i
        elif (char.isspace() or i == len(text) - 1) and in_word:
            end_pos = i if char.isspace() else i + 1
            word = text[word_start:end_pos]
            result.append(f"<{word_start}>{word}<{end_pos-1}>")
            in_word = False
            if char.isspace():
                result.append(char)
    
    return "".join(result)
# text = "David Sandburg was born in Stockholm, Sweden."
# output = add_token_positions(text)
# print(output)

In [32]:
class HallucinationPromptSystem:
    def __init__(self):
        self.base_task = """
        Task: Analyze the generated text and identify potential hallucinations by:
        1. Compare the generated text with the given context
        2. Mark spans that contain information not supported by the context
        3. Assign probability scores (0.0-1.0) to potential hallucinations
        4. Return the results in JSON format with start/end positions
        """
    
    def n_shot_prompt(self, input_text, context, generated_text, n_shots=1):
        examples = [
            {
                "context": "The Olympic Games are a major international sports competition.",
                "generated": "Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.",
                "output": """{"soft_labels": [{"start": 25, "end": 31, "prob": 0.9}, {"start": 45, "end": 49, "prob": 1.0}, {"start": 69, "end": 83, "prob": 0.9}]}""",
                "explanation": "The specific medal, year and location details are not supported by context."
            },
            {
                "context": "Erysiphales is an order of fungi.",
                "generated": "The Elysiphale order contains 5 genera.",
                "output": """{"soft_labels": [{"start": 30, "end": 31, "prob": 1.0}]}""",
                "explanation": "The specific number of genera is not mentioned in context."
            },
            {
                "context": "Arthropods are a phylum of invertebrate animals.",
                "generated": "Yes, all arachnids have antennas. However, not all of them are visible to the naked eye.",
                "output": """{"soft_labels": [{"start": 0, "end": 3, "prob": 0.6}, {"start": 9, "end": 18, "prob": 0.6}, {"start": 63, "end": 70, "prob": 0.7}, {"start": 78, "end": 87, "prob": 0.7}]}""",
                "explanation": "The statement about all arachnids and visibility is not supported by context."
            }
        ][:n_shots]

        prompt = f"Here are {n_shots} examples of hallucination detection:\n\n"
        for i, example in enumerate(examples, 1):
            prompt += f"""
            Example {i}:
            Context: {example['context']}
            Generated: {example['generated']}
            Generated with positions: {add_token_positions(example['generated'])}
            Output: {example['output']}
            Explanation: {example['explanation']}\n\n"""
        
        prompt += f"""
        Now analyze this case:
        Context: {context}
        Generated Text: {generated_text}
        Generated with positions: {add_token_positions(generated_text)}
        
        {self.base_task}
        """
        return prompt

    def chain_of_thought_prompt(self, input_text, context, generated_text):
        """Chain of thought approach encouraging step-by-step reasoning"""
        prompt = f"""Context: {context}
                Generated Text: {generated_text}

                Let's analyze this step by step:
                1. First, break down the generated text into individual claims
                2. For each claim:
                - Check if it's explicitly supported by the context
                - Check if it can be reasonably inferred from the context
                - Mark any information that goes beyond the context
                3. For identified hallucinations:
                - Note the exact span (start and end positions)
                - Assess probability based on how far it deviates from context
                4. Format the results as JSON

                Example analysis:
                Context: "Albert Einstein was a physicist who developed the theory of relativity."
                Generated: "Albert Einstein was born in Germany in 1879 and won the Nobel Prize in Physics in 1921 for his work on quantum mechanics."

                Analysis:
                1. Claims:
                - Albert Einstein was born in Germany in 1879 (not in context)
                - Won Nobel Prize in Physics in 1921 (not in context)
                - Work on quantum mechanics (not in context, also misleading)
                2. Hallucination assessment:
                - "was born in Germany in 1879" (spans 15-41) not supported
                - "won the Nobel Prize in Physics in 1921" (spans 46-82) not supported
                - "quantum mechanics" (spans 90-106) is misleading as Einstein's Nobel Prize was for photoelectric effect
                3. Final output:
                {{"soft_labels": [
                    {{"start": 15, "end": 41, "prob": 0.8}},
                    {{"start": 46, "end": 82, "prob": 0.7}},
                    {{"start": 90, "end": 106, "prob": 0.9}}
                ]}}

                Now analyze the given case following these steps:"""
        return prompt

    def generate_prompt(self, approach, input_text, context, generated_text, n_shots=1):
        """Generate prompt based on specified approach"""
        if approach == "zero-shot":
            return self.n_shot_prompt(input_text, context, generated_text, n_shots=0)
        elif approach == "one-shot":
            return self.n_shot_prompt(input_text, context, generated_text, n_shots=1)
        elif approach == "three-shot":
            return self.n_shot_prompt(input_text, context, generated_text, n_shots=3)
        elif approach == "chain-of-thought":
            return self.chain_of_thought_prompt(input_text, context, generated_text)
        else:
            raise ValueError("Invalid approach specified")
        
prompt_system = HallucinationPromptSystem()


In [47]:
def prompt_fix(row):   
    output = []
    input_text = row["query"]
    context= row['scraping_and_procesor']
    generated_text= row['response']
    # respuesta = prompt_system.generate_prompt("chain-of-thought", input_text, context, generated_text)
    respuesta = prompt_system.generate_prompt("one-shot", input_text, context, generated_text)
    # respuesta = prompt_system.generate_prompt("three-shot", input_text, context, generated_text)
    print(respuesta)
    output.append(respuesta)
    return output

df["propmt"] = df.apply(prompt_fix, axis=1)


Here are 1 examples of hallucination detection:


            Example 1:
            Context: The Olympic Games are a major international sports competition.
            Generated: Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.
            Generated with positions: <0>Petra<4> <6>van<8> <10>Stoveren<17> <19>won<21> <23>a<23> <25>silver<30> <32>medal<36> <38>in<39> <41>the<43> <45>2008<48> <50>Summer<55> <57>Olympics<64> <66>in<67> <69>Beijing,<76> <78>China.<83>
            Output: {"soft_labels": [{"start": 25, "end": 31, "prob": 0.9}, {"start": 45, "end": 49, "prob": 1.0}, {"start": 69, "end": 83, "prob": 0.9}]}
            Explanation: The specific medal, year and location details are not supported by context.


        Now analyze this case:
        Context: ['He was part of two championship winning sides in 1961 and 1966. He played for Chile in two World Cups; the 1962 and 1966.', 'Stats of Alberto Fouillioux ; World Cup qualification South Am

In [34]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:5000/v1",
                api_key="lm-studio")

In [35]:

def model_llm(query):
    client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="API_KEY",
    )

    completion = client.chat.completions.create(
    model="deepseek/deepseek-r1-distill-llama-70b",
    messages = [
        {"role": "system", "content": "Analyze text and call store_hallucination_analysis with results"},
        {"role": "user", "content": query}
    ]
    )
    print(completion.choices[0].message.content)
    return completion.choices[0].message.content

In [36]:
import re
import json
from openai import OpenAI

def extract_json_from_text(text):
    """
    Improved JSON extraction with better error handling
    """
    try:
        # Look for complete JSON blocks with proper formatting
        json_match = re.search(r'{\s*"soft_labels":\s*\[.*?\]\s*}', text, re.DOTALL)
        if json_match:
            json_str = json_match.group()
            # Clean up any residual markdown formatting
            json_str = re.sub(r'```json\n?', '', json_str)
            return json.loads(json_str)
        return None
    except Exception as e:
        print(f"Error extracting JSON: {str(e)}")
        return None

def llm(query):
    print(query[0])
    # messages = [
    #     {"role": "system", "content": "Analyze text and call store_hallucination_analysis with results"},
    #     {"role": "user", "content": query[0]}
    # ]

    try:

        model_response = model_llm(query[0])
        result = extract_json_from_text(model_response)
        
        # Validate the result structure
        if result and "soft_labels" in result:
            for label in result["soft_labels"]:
                if not all(k in label for k in ["start", "end", "prob"]):
                    return {"soft_labels": []}
            return result
        return {"soft_labels": []}
        
    except Exception as e:
        print(f"Error: {str(e)}")
        return {"soft_labels": []}


In [48]:
# To fix the SettingWithCopyWarning
df_copy = df[:].copy()  # Use .copy() to avoid modifying a slice

df_copy["generation"] = df_copy["propmt"].apply(llm)

Here are 1 examples of hallucination detection:


            Example 1:
            Context: The Olympic Games are a major international sports competition.
            Generated: Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.
            Generated with positions: <0>Petra<4> <6>van<8> <10>Stoveren<17> <19>won<21> <23>a<23> <25>silver<30> <32>medal<36> <38>in<39> <41>the<43> <45>2008<48> <50>Summer<55> <57>Olympics<64> <66>in<67> <69>Beijing,<76> <78>China.<83>
            Output: {"soft_labels": [{"start": 25, "end": 31, "prob": 0.9}, {"start": 45, "end": 49, "prob": 1.0}, {"start": 69, "end": 83, "prob": 0.9}]}
            Explanation: The specific medal, year and location details are not supported by context.


        Now analyze this case:
        Context: ['He was part of two championship winning sides in 1961 and 1966. He played for Chile in two World Cups; the 1962 and 1966.', 'Stats of Alberto Fouillioux ; World Cup qualification South Am

In [49]:
df_copy["generation"][0]

{'soft_labels': [{'start': 5, 'end': 17, 'prob': 1.0}]}

In [50]:
df = df_copy

In [51]:
import json
from tqdm import tqdm
example = {"lang": "EN", "model_input": "Did Alberto Fouillioux ever play in a world cup championship?", "id": "tst-en-1", "model_output_text": " No, Albero Foulois was not in any of the FIFA World Cup finals.\n", "hard_labels": [[1, 3], [5, 11], [12, 19], [20, 23], [24, 27], [42, 46], [57, 63]], "soft_labels": [{"start": 1, "end": 3, "prob": 0.2619964927434921}, {"start": 5, "end": 11, "prob": 0.3255588561296463}, {"start": 12, "end": 19, "prob": 0.3380719870328903}, {"start": 20, "end": 23, "prob": 0.3233960121870041}, {"start": 24, "end": 27, "prob": 0.2605454623699188}, {"start": 42, "end": 46, "prob": 0.222648024559021}, {"start": 57, "end": 63, "prob": 0.32610486447811127}]}

def generate_ouput(row):
        # Create the JSON object with relevant information
    json_object = {
            "lang": "EN",
            "model_input": row["query"],
            "id": row["id"],
            "model_output_text": row["response"],
            "soft_labels": row["generation"]["soft_labels"],
        }
    return json_object

df["output"] = df.apply(generate_ouput, axis=1)

In [52]:
def create_hard_labels(output_dict):
    hard_labels = []
    for soft_label in output_dict["soft_labels"]:
        dict_prob = {}
        dict_prob["start"] = soft_label["start"]
        dict_prob["end"] = soft_label["end"]
        hard_labels.append(dict_prob )
    output_dict["hard_label"] = hard_labels
    return output_dict


In [53]:
df["output_v1"] = df["output"].apply(create_hard_labels)

In [54]:
# Save as JSON
responses = []
for row in df["output_v1"]:
    responses.append(row)

# with open("responses.json", "w", encoding="utf-8") as json_file:
#     json.dump(responses, json_file, ensure_ascii=False, indent=2)

# Save as JSONL
with open("responses.jsonl", "w", encoding="utf-8") as jsonl_file:
    for item in responses:
        json.dump(item, jsonl_file, ensure_ascii=False)
        jsonl_file.write("\n")

print("Responses have been saved to responses.json and responses.jsonl")

Responses have been saved to responses.json and responses.jsonl


In [55]:
# import json
# from tqdm import tqdm
# init = 0
# end = 5
# data_val_all_cut = data_val_all[init:end]
# output_cut = output[init:end]
# responses = []
# for num_sample, (prompt, sample) in enumerate(zip(tqdm(output, desc="Processing prompts"), data_val_all_cut)):
#     template = f"<s>[INST] {prompt} [/INST]"
#     response = lcpp_llm.invoke(template)
#     result=extract_soft_labels(response)
#     print(f"json----{result}")
#     # Parse the response string into a Python dictionary
#     try:
#         response_dict = result
#     except json.JSONDecodeError:
#         print(f"Error decoding JSON for sample {num_sample}. Response: {response}")
#         response_dict = {"soft_labels": []}

#     # Create the JSON object with relevant information
#     json_object = {
#         "id": f'val-es-{num_sample}',
#         "soft_labels": response_dict.get('soft_labels', []),
#         "model_output_text": sample['model_output_text']
#     }
    
#     responses.append(json_object)

# # Save as JSON
# with open("responses.json", "w", encoding="utf-8") as json_file:
#     json.dump(responses, json_file, ensure_ascii=False, indent=2)

# # Save as JSONL
# with open("responses.jsonl", "w", encoding="utf-8") as jsonl_file:
#     for item in responses:
#         json.dump(item, jsonl_file, ensure_ascii=False)
#         jsonl_file.write("\n")

# print("Responses have been saved to responses.json and responses.jsonl")

In [57]:
import json
from pathlib import Path

def validate_label_ranges(entry, text_length):
    """Validate and clean label ranges based on text length."""
    valid_soft = []
    valid_hard = []
    
    # Validate soft labels
    for label in entry.get('soft_labels', []):
        try:
            start = int(label.get('start', -1))
            end = int(label.get('end', -1))
            
            # Check if ranges are valid
            if (0 <= start < end <= text_length and 
                'prob' in label and 
                isinstance(label['prob'], (int, float)) and 
                0 <= label['prob'] <= 1):
                valid_soft.append({
                    'start': start,
                    'end': end,
                    'prob': float(label['prob'])
                })
        except (TypeError, ValueError):
            continue
    
    # Validate hard labels
    for label in entry.get('hard_label', []):
        try:
            start = int(label.get('start', -1))
            end = int(label.get('end', -1))
            
            # Check if ranges are valid
            if 0 <= start < end <= text_length:
                valid_hard.append({
                    'start': start,
                    'end': end
                })
        except (TypeError, ValueError):
            continue
    
    entry['soft_labels'] = valid_soft
    entry['hard_label'] = valid_hard
    return entry

def clean_jsonl_file():
    input_file = Path('responses.jsonl')
    output_file = Path('cleaned_responses.jsonl')
    
    with open(input_file, encoding='utf-8') as f_in, open(output_file, 'w', encoding='utf-8') as f_out:
        for line in f_in:
            try:
                entry = json.loads(line.strip())
                
                # Get text length for validation
                text_length = len(entry.get('model_output_text', ''))
                
                # Clean and validate the entry
                cleaned = validate_label_ranges(entry, text_length)
                
                # Write the cleaned entry
                f_out.write(json.dumps(cleaned, ensure_ascii=False) + '\n')
                
                # Print info for problematic entries (like tst-en-5)
                if entry['id'] == 'tst-en-5':
                    print(f"Cleaned entry {entry['id']}:")
                    print(f"Text length: {text_length}")
                    print(f"Original labels: {entry.get('soft_labels', [])}")
                    print(f"Cleaned labels: {cleaned['soft_labels']}")
                    
            except json.JSONDecodeError as e:
                print(f"Error processing line: {e}")
                continue

if __name__ == "__main__":
    clean_jsonl_file()

Cleaned entry tst-en-5:
Text length: 186
Original labels: [{'start': 24, 'end': 42, 'prob': 1.0}, {'start': 57, 'end': 65, 'prob': 0.9}, {'start': 75, 'end': 83, 'prob': 0.85}, {'start': 173, 'end': 185, 'prob': 0.8}]
Cleaned labels: [{'start': 24, 'end': 42, 'prob': 1.0}, {'start': 57, 'end': 65, 'prob': 0.9}, {'start': 75, 'end': 83, 'prob': 0.85}, {'start': 173, 'end': 185, 'prob': 0.8}]
